# 03 · Results of the spatial CV

* Scores per outer fold, then mean, SD and minimum over the 5 folds: **RMSE, MAE, R², bias, Spearman**.
* **Skill vs depth-only** = 1 − RMSE / RMSE of the depth reference (> 0: location adds information beyond depth).
* Scores by distance to the nearest training borehole and by depth band.
* **Model choice per target** (rule fixed before the results): lowest mean fold RMSE; among models within one standard error, the lowest fold SD, then the lowest MAE. The depth reference competes; if it is chosen, the 4 inputs add nothing beyond depth for that parameter.

All tables here are aggregates (no borehole codes or locations). **Run time:** under 1 minute.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])

In [ ]:
O = H.out_dir(S, "03_results")
preds = pd.read_csv(H.out_dir(S, "02_spatial_cv", private=True) / "outer_predictions.csv")
main_block = H.block_sizes(S)[0]
folds = H.fold_scores(preds); folds.to_csv(O / "fold_scores.csv", index=False)
summ = H.summary(folds); summ.to_csv(O / "summary.csv", index=False)
pooled = H.score_table(preds, H.RUN_KEYS); pooled.to_csv(O / "pooled_scores.csv", index=False)
cols = ["target", "block_m", "model", "rmse_mean", "rmse_sd", "mae_mean", "r2_mean", "bias_mean", "spearman_mean", "skill_vs_depth"]
display(summ[cols].round(3))

## Model choice per target (main block)

In [ ]:
choice = pd.DataFrame([H.select_model(summ, t, main_block) for t in summ.target.unique()])
choice.to_csv(O / "model_choice.csv", index=False)
display(choice)

## By distance to the nearest training borehole, and by depth

In [ ]:
dist = H.distance_table(preds); dist.to_csv(O / "scores_by_distance.csv", index=False)
dep = H.depth_table(preds); dep.to_csv(O / "scores_by_depth.csv", index=False)
display(dist[dist.block_m == main_block].pivot_table(index=["target", "distance_bin"], columns="model", values="rmse").round(2))
display(dep[dep.block_m == main_block].pivot_table(index=["target", "depth_band"], columns="model", values="rmse").round(2))

## Figures

In [ ]:
ts = list(summ.target.unique())

m = preds[preds.block_m == main_block]
models = list(m.model.unique())
fig, axes = plt.subplots(len(ts), len(models), figsize=(3 * len(models), 3 * len(ts)), squeeze=False)
for i, t in enumerate(ts):
    for j, mod in enumerate(models):
        g = m[(m.target == t) & (m.model == mod)]
        ax = axes[i, j]
        ax.scatter(g.y, g.y_pred, s=4, alpha=0.3)
        lim = [min(g.y.min(), g.y_pred.min()), max(g.y.max(), g.y_pred.max())]
        ax.plot(lim, lim, "k--", lw=0.8)
        ax.set_title(f"{t} · {mod}", fontsize=9); ax.set_xlabel("observed"); ax.set_ylabel("predicted")
plt.tight_layout(); plt.savefig(O / "observed_vs_predicted_main_block.png", dpi=200); plt.show()
print(f"Finished in {time.time() - T0:.0f} s. Next: notebook 04.")